# Agent Goal Accuracy Evaluation + Combined Metric for Derived KPIs


## Imports & Functions

In [ ]:
import asyncio
import sys
import os
from pathlib import Path
import dotenv
import logging
from typing import List, Dict, Optional, Any
import json

# Ensure we're in the right directory and add to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # Go up to project root
os.chdir(str(project_root))
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {project_root}")

from agent.graph_utils import initialize_graph, test_graph_connection, close_graph
from agent.vector_db_utils import initialize_database, close_database

from agent.single_agent import single_agent_run, single_agent_run_input_list
from agent.multi_agent_router.orchestrator import (
    multi_agent_router_orchestrator_run, 
    multi_agent_router_orchestrator_run_input_list,

)

print("All imports successful")

In [ ]:
import asyncio
from openai import AsyncOpenAI
from ragas.llms.base import llm_factory
from ragas.metrics.collections import AgentGoalAccuracyWithoutReference, AgentGoalAccuracyWithReference
from ragas.messages import AIMessage, HumanMessage, ToolCall, ToolMessage

In [ ]:
# Setup LLM
client_openai = AsyncOpenAI()
model_gpt_4 = "gpt-4o-mini"
model_gpt_5mini = "gpt-5-mini"
llm = llm_factory(model_gpt_5mini, client=client_openai, max_tokens=8000)

In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd

def write_csv(obj, filename: str, out_dir: str | Path | None = None, *, index: bool = False) -> Path:
    """Write `obj` as a CSV into evaluation/ragas/ (relative to project_root if available).
    
    `obj` can be a pandas DataFrame, or something coercible into one (list[dict], dict, etc.).
    Returns the written file path.
    """
    if isinstance(obj, pd.DataFrame):
        df_to_write = obj
    else:
        df_to_write = pd.DataFrame(obj)

    base_dir = None
    if out_dir is not None:
        base_dir = Path(out_dir)
    elif "project_root" in globals():
        base_dir = Path(project_root) / "evaluation" / "agent_goal_accuracy"
    else:
        base_dir = Path.cwd() / "evaluation" / "agent_goal_accuracy"

    base_dir.mkdir(parents=True, exist_ok=True)
    if not filename.lower().endswith(".csv"):
        filename = f"{filename}.csv"

    path = base_dir / filename
    df_to_write.to_csv(path, index=index)
    print(f"Wrote CSV: {path} ({len(df_to_write)} rows)")
    return path

# Experiment Agent Goal Accuracy

In [ ]:
from ragas import experiment, Dataset
from agent.multi_agent_router.orchestrator import multi_agent_router_orchestrator_run_input_list
from dataclasses import is_dataclass, fields as dataclass_fields
from typing import Tuple

# -----------------------------
# Helpers
# -----------------------------

def kpi_queries_and_expected_names(
    kpi_obj: Any,
    *,
    strip_year_suffix: bool = True,
) -> Tuple[List[str], List[str]]:
    """
    Return (queries, expected_kpi_names) aligned by dataclass field order.

    For the quantitative KPI dataclasses, fields are often named like `EBITDA_0/1/2`.
    With `strip_year_suffix=True` this becomes `EBITDA` for all three queries.
    """
    if not is_dataclass(kpi_obj):
        raise TypeError(f"Expected a dataclass instance, got: {type(kpi_obj)!r}")

    queries: List[str] = []
    expected_names: List[str] = []
    for field in dataclass_fields(kpi_obj):
        if field.name in ("client", "_built"):
            continue
        value = getattr(kpi_obj, field.name)
        if value is None:
            continue
        name = field.name
        if strip_year_suffix and len(name) > 2 and name[-2] == "_" and name[-1].isdigit():
            name = name[:-2]
        expected_names.append(name)
        queries.append(str(value))
    return queries, expected_names


# -----------------------------
# Dataset Preparation for Agent Goal Accuracy
# -----------------------------

def prepare_dataset_agent_goal(
    queries: List[str],
    expected_answers: List[str],
    ragas_messages_list: List[List[Any]],
    kpi_names: List[str],
    years: List[Any],
    name: str = "agent_goal_evaluation",
    kpi_names_expected: Optional[List[str]] = None,
    kpi_names_agent: Optional[List[str]] = None,
    value_answers: Optional[List[str]] = None,
    response_answers: Optional[List[str]] = None,
    num_tool_calls: Optional[List[int]] = None,
) -> Dataset:
    """
    Prepare a RAGAS dataset for agent goal accuracy evaluation.

    Args:
        queries: List of original queries
        expected_answers: List of expected/reference answers
        ragas_messages_list: List of RAGAS message lists (one per query)
        kpi_names: List of (chosen) KPI names for grouping/analysis
        years: List of fiscal years
        name: Dataset name
        kpi_names_expected: Optional stable KPI names (e.g., derived from dataclass fields)
        kpi_names_agent: Optional KPI names returned by the agent
        value_answers: Optional extracted value answer strings
        num_tool_calls: Optional list of tool call counts per query
    Returns:
        RAGAS Dataset object
    """
    n = len(queries)
    if len(expected_answers) != n:
        raise ValueError(f"Length mismatch: expected_answers={len(expected_answers)} vs queries={n}")
    if len(ragas_messages_list) != n:
        raise ValueError(f"Length mismatch: ragas_messages_list={len(ragas_messages_list)} vs queries={n}")
    if len(kpi_names) != n:
        raise ValueError(f"Length mismatch: kpi_names={len(kpi_names)} vs queries={n}")
    if len(years) != n:
        raise ValueError(f"Length mismatch: years={len(years)} vs queries={n}")

    if kpi_names_expected is None:
        kpi_names_expected = [""] * n
    elif len(kpi_names_expected) != n:
        raise ValueError(f"Length mismatch: kpi_names_expected={len(kpi_names_expected)} vs queries={n}")

    if kpi_names_agent is None:
        kpi_names_agent = [""] * n
    elif len(kpi_names_agent) != n:
        raise ValueError(f"Length mismatch: kpi_names_agent={len(kpi_names_agent)} vs queries={n}")

    if value_answers is None:
        value_answers = [""] * n
    elif len(value_answers) != n:
        raise ValueError(f"Length mismatch: value_answers={len(value_answers)} vs queries={n}")
    
    if response_answers is None:
        response_answers = [""] * n
    elif len(response_answers) != n:
        raise ValueError(f"Length mismatch: response_answers={len(response_answers)} vs queries={n}")
    
    if num_tool_calls is None:
        num_tool_calls = [0] * n
    elif len(num_tool_calls) != n:
        raise ValueError(f"Length mismatch: num_tool_calls={len(num_tool_calls)} vs queries={n}")

    dataset = Dataset(
        name=name,
        backend="local/csv",
        root_dir="./data/agent_goal_evaluation",
    )

    for i in range(n):
        expected_kpi_name = kpi_names_expected[i] or ""
        agent_kpi_name = kpi_names_agent[i] or ""
        chosen_kpi_name = kpi_names[i] or expected_kpi_name or agent_kpi_name or ""
        dataset.append({
            "kpi_name": chosen_kpi_name,
            "kpi_name_expected": expected_kpi_name,
            "kpi_name_agent": agent_kpi_name,
            "year": years[i],
            "value_answer": value_answers[i],
            "response_answer": response_answers[i],
            "expected_answer": expected_answers[i],
            "query": queries[i],
            "ragas_messages": ragas_messages_list[i],
            "num_tool_calls": num_tool_calls[i],
            
            
        })

    dataset.save()
    return dataset


async def agent_run_with_history(
    query_list: List[str],
    expected_answers: List[str],
    years: Optional[List[Any]] = None,
    expected_kpi_names: Optional[List[str]] = None,
    name: str = "agent_goal_evaluation",
    client: str = "default_client",
    reranker_entity: str = "RRF",
    reranker_fact: str = "RRF",
    limit_vector_results: int = 15,
    alpha_hybrid: float = 0.6,
    limit_fact: int = 30,
    limit_entity: int = 10,
    k: int = 2,
) -> Dataset:
    """
    Run the SINGLE agent for each query and prepare the dataset for agent goal accuracy evaluation.
    """
    logger = logging.getLogger(__name__)

    if expected_kpi_names is not None and len(expected_kpi_names) != len(query_list):
        raise ValueError(
            f"Length mismatch: expected_kpi_names has {len(expected_kpi_names)} entries, "
            f"but query_list has {len(query_list)} queries."
        )

    # If years not provided, create empty list
    if years is None:
        years = [""] * len(query_list)

    try:
        await initialize_graph()
        logger.info("Graph database initialized")
        graph_ok = await test_graph_connection()
        if not graph_ok:
            logger.error("Graph database connection test failed.")
    except Exception as e:
        logger.error(f"Failed to initialize graph database: {e}")
        raise

    try:
        initialize_database()
        logger.info("Vector database initialized")
    except Exception as e:
        logger.error(f"Failed to initialize vector database: {e}")
        raise

    # Create agent runner
    agent_runner = single_agent_run_input_list(
        client=client,
        reranker_entity=reranker_entity,
        reranker_fact=reranker_fact,
        limit_vector_results=limit_vector_results,
        alpha_hybrid=alpha_hybrid,
        limit_fact=limit_fact,
        limit_entity=limit_entity,
        k=k,
    )

    queries: List[str] = []
    ragas_messages_list: List[List[Any]] = []
    kpi_names: List[str] = []
    kpi_names_expected: List[str] = []
    kpi_names_agent: List[str] = []
    years_list: List[Any] = []
    value_answers: List[str] = []
    response_answers: List[str]= []
    num_tool_calls_list: List[int] = []

    for idx, (query, year) in enumerate(zip(query_list, years)):
        print(f"Processing query: {query[:70]}...")

        expected_kpi_name = (expected_kpi_names[idx] if expected_kpi_names is not None else "") or ""
        result = await agent_runner.run_with_full_history([query], expected_kpi_name)

        # Convert to RAGAS format - now including final_output for the agent's answer
        ragas_msgs = agent_runner.convert_to_ragas_messages(
            result.get("input_list", []),
            final_output=result.get("final_output", {}),  # Include the final answer!
        )

        # Count tool calls in the ragas messages
        tool_call_count = sum(1 for msg in ragas_msgs if isinstance(msg, AIMessage) and msg.tool_calls)
        
        agent_kpi_name = (result.get("kpi_name", "") or "")
        chosen_kpi_name = expected_kpi_name or agent_kpi_name or ""

        queries.append(query)
        ragas_messages_list.append(ragas_msgs)
        kpi_names.append(chosen_kpi_name)
        kpi_names_expected.append(expected_kpi_name)
        kpi_names_agent.append(agent_kpi_name)
        # Use year from agent result if available, otherwise use provided year
        years_list.append(result.get("year", "") or year)
        value_answers.append(result.get("final_output", {}).get("value", "") or "")
        response_answers.append(result.get("final_output", {}).get("response", "") or "")
        num_tool_calls_list.append(tool_call_count)


        # Debug: print last message to verify final answer is included
        if ragas_msgs:
            print(f"  -> Final AI message: {str(ragas_msgs[-1])[:200]}...")

    await close_graph()
    close_database()
    logger.info("Databases closed")

    # Prepare dataset
    dataset = prepare_dataset_agent_goal(
        queries=queries,
        expected_answers=expected_answers,
        ragas_messages_list=ragas_messages_list,
        kpi_names=kpi_names,
        kpi_names_expected=kpi_names_expected,
        kpi_names_agent=kpi_names_agent,
        years=years_list,
        name=name,
        value_answers=value_answers,
        response_answers=response_answers,
        num_tool_calls=num_tool_calls_list,
    )

    print(f"Dataset prepared with {len(queries)} samples")
    return dataset


async def multi_agent_run_with_history(
    query_list: List[str],
    expected_answers: List[str],
    years: Optional[List[Any]] = None,
    expected_kpi_names: Optional[List[str]] = None,
    name: str = "multi_agent_goal_evaluation",
    client: str = "default_client",
    reranker_entity: str = "RRF",
    reranker_fact: str = "RRF",
    limit_vector_results: int = 15,
    alpha_hybrid: float = 0.6,
    limit_fact: int = 30,
    limit_entity: int = 10,
    k: int = 2,
) -> Dataset:
    """
    Run the MULTI-AGENT for each query and prepare the dataset for agent goal accuracy evaluation.
    """
    logger = logging.getLogger(__name__)

    if expected_kpi_names is not None and len(expected_kpi_names) != len(query_list):
        raise ValueError(
            f"Length mismatch: expected_kpi_names has {len(expected_kpi_names)} entries, "
            f"but query_list has {len(query_list)} queries."
        )

    # If years not provided, create empty list
    if years is None:
        years = [""] * len(query_list)

    try:
        await initialize_graph()
        logger.info("Graph database initialized")
        graph_ok = await test_graph_connection()
        if not graph_ok:
            logger.error("Graph database connection test failed.")
    except Exception as e:
        logger.error(f"Failed to initialize graph database: {e}")
        raise

    try:
        initialize_database()
        logger.info("Vector database initialized")
    except Exception as e:
        logger.error(f"Failed to initialize vector database: {e}")
        raise

    # Create multi-agent runner
    agent_runner = multi_agent_router_orchestrator_run_input_list(
        client=client,
        reranker_entity=reranker_entity,
        reranker_fact=reranker_fact,
        limit_vector_results=limit_vector_results,
        alpha_hybrid=alpha_hybrid,
        limit_fact=limit_fact,
        limit_entity=limit_entity,
        k=k,
    )

    queries: List[str] = []
    ragas_messages_list: List[List[Any]] = []
    kpi_names: List[str] = []
    kpi_names_expected: List[str] = []
    kpi_names_agent: List[str] = []
    years_list: List[Any] = []
    value_answers: List[str] = []
    response_answers: List[str] = []
    num_tool_calls_list: List[int] = []

    for idx, (query, year) in enumerate(zip(query_list, years)):
        print(f"Processing query: {query[:70]}...")
        expected_kpi_name = (expected_kpi_names[idx] if expected_kpi_names is not None else "") or ""

        result = await agent_runner.run_with_full_history([query],expected_kpi_name )

        # Convert to RAGAS format - including final_output for the agent's answer
        ragas_msgs = agent_runner.convert_to_ragas_messages(
            result.get("input_list", []),
            final_output=result.get("final_output", {}),
        )

        # Count tool calls in the ragas messages
        tool_call_count = sum(1 for msg in ragas_msgs if isinstance(msg, AIMessage) and msg.tool_calls)

        agent_kpi_name = (result.get("kpi_name", "") or "")
        chosen_kpi_name = expected_kpi_name or agent_kpi_name or ""

        queries.append(query)
        ragas_messages_list.append(ragas_msgs)
        kpi_names.append(chosen_kpi_name)
        kpi_names_expected.append(expected_kpi_name)
        kpi_names_agent.append(agent_kpi_name)
        # Use year from agent result if available, otherwise use provided year
        years_list.append(result.get("year", "") or year)
        value_answers.append(result.get("final_output", {}).get("value", "") or "")
        # Multi-agent uses "report" key instead of "response"
        report_text = result.get("final_output", {}).get("report", "") or ""
        response_answers.append(report_text.replace("\n", " ") if report_text else "")
        num_tool_calls_list.append(tool_call_count)

        # Debug: print last message to verify final answer is included
        if ragas_msgs:
            print(f"  -> Final AI message: {str(ragas_msgs[-1])[:200]}...")

    await close_graph()
    close_database()
    logger.info("Databases closed")

    # Prepare dataset
    dataset = prepare_dataset_agent_goal(
        queries=queries,
        expected_answers=expected_answers,
        ragas_messages_list=ragas_messages_list,
        kpi_names=kpi_names,
        kpi_names_expected=kpi_names_expected,
        kpi_names_agent=kpi_names_agent,
        years=years_list,
        name=name,
        value_answers=value_answers,
        response_answers=response_answers,
        num_tool_calls=num_tool_calls_list,
    )

    print(f"Multi-Agent Dataset prepared with {len(queries)} samples")
    return dataset

In [ ]:
@experiment()
async def run_evaluation_agent_goal_accuracy(row: Dict[str, Any], name: str = "agent_goal_accuracy") -> Dict[str, Any]:
    """
    Evaluate agent goal accuracy for a single row.

    """

    print("Evaluating row:", row)

    metric_agent_goal_accuracy = AgentGoalAccuracyWithReference(llm=llm)
    result_agent_goal_accuracy = await metric_agent_goal_accuracy.ascore(
        user_input=row.get("ragas_messages", []),
        reference=row.get("expected_answer", ""),
    )
    if row.get("value_answer", "") =="": agent_answer =  row.get("response_answer", "") 
    else: agent_answer = row.get("value_answer", "")
    return {
        "kpi_name": row.get("kpi_name", ""),
        "kpi_name_expected": row.get("kpi_name_expected", ""),
        "kpi_name_agent": row.get("kpi_name_agent", ""),
        "year": row.get("year", ""),
        "agent_goal_accuracy": float(result_agent_goal_accuracy.value),
        "agent_answer": agent_answer,
        "agent_value": row.get("value_answer", ""),
        "agent_response": row.get("response_answer", ""),
        "expected_answer": row.get("expected_answer", ""),
        "num_tool_calls": row.get("num_tool_calls", 0),
        "experiment_name": name,
    }

In [ ]:
import pandas as pd

def create_quantitative_kpi_dataframe(
    client_company: str,
    years: List[int],
    expected_answers: Optional[Dict[str, Dict[int, str]]] = None
) -> pd.DataFrame:
    """
    Create a DataFrame with all quantitative KPIs from the speedboat framework.
    
    Args:
        client_company: The company name (e.g., "RWE", "Walmart")
        years: List of fiscal years (e.g., [2024, 2023, 2022])
        expected_answers: Optional dict mapping kpi_name -> {year: expected_value}
                         Example: {"Total Sales Revenue": {2024: "EUR 24,224 million", 2023: "EUR 22,000 million"}}
    
    Returns:
        DataFrame with columns: kpi_name, year, client, query, expected_answer
    """
    # Define the quantitative KPIs from the speedboat framework
    kpi_names = [
        "Total Sales Revenue",
        "Gross Profit",
        "EBITDA",
        "EBIT",
        "Net Income",
        "Total Cash",
        "Total Equity",
        "Tangible Net Worth",
        "Capital Expenditures (Capex)",
        "Free Cash Flow",
        "Unrestricted Cash",
        "Undrawn Loan Facilities",
    ]
    
    # Create rows for the DataFrame
    rows = []
    for kpi in kpi_names:
        for year in years:
            # Get expected answer if provided
            expected = ""
            if expected_answers and kpi in expected_answers:
                expected = expected_answers[kpi].get(year, "")
            
            rows.append({
                "kpi_name": kpi,
                "year": year,
                "client": client_company,
                "query": f"Provide the {kpi} for the fiscal year {year} for {client_company}.",
                "expected_answer": expected
            })
    
    df = pd.DataFrame(rows)
    print(f"Created DataFrame with {len(df)} rows ({len(kpi_names)} KPIs x {len(years)} years) for {client_company}")
    return df




## EVAL QUANTITATIVE With Reference

In [ ]:
expected_answers_RWE={
        "Total Sales Revenue": {
            2024:"EUR 24,224 million or EUR 24,439 million.",
            2023: "EUR 28,521 million or EUR 28,689 million.",
            2022: "EUR 38,415 million"
        },
        "Gross Profit": {
            2024: "EUR 9,031 million or EUR 8,816 million.",
            2023: "EUR 11,362 million or EUR 11,530 million.",
            2022: "Not provided: Cost of materials only provided for 2024 and 2023."
        },
        "EBITDA": {
            2024: "EUR 5,680 million",
            2023: "EUR 7,749 million",
            2022: "EUR 6,310 million",
        },
         "EBIT": {
            2024: "EUR 3,561 million or EUR 6,324 million.",
            2023:  "EUR 5,802 million or EUR 4,442 million",
            2022:  "EUR 4,568 million",
        },
         "Net Income": {
            2024: "EUR 5,135 million or EUR 2,322 million",
            2023: "EUR 1,515 million or EUR 4,098 million",
            2022: "EUR 2,717 million or 3,253 million",
        },
         "Total Cash": {
            2024: "EUR 5,090 million",
            2023: "EUR 6,917 million",
            2022: "Not available for 2022",
        },
        "Total Equity": {
            2024: "EUR 33,623 million",
            2023:  "EUR 33,604 million",
            2022: "EUR 29,304 million",
        },
        "Tangible Net Worth": {
            2024: "EUR 23,373 million",
            2023:  "EUR 23,817 million",
            2022: "not provided",
        },
        "Capital Expenditures (Capex)": {
            2024: "EUR 11,240 million",
            2023:  "EUR 9,979 million",
            2022: "not provided",
        },
        "Free Cash Flow": {
            2024: "-€4,106 million",
            2023:  "−€4,594 million",
            2022: "EUR -1,968 million",
        },
        "Unrestricted Cash": {
            2024: "EUR 5,090 million or not provided",
            2023: "EUR 6,917 million or not provided",
            2022: "not provided",
        },
        "Undrawn Loan Facilities": {
            2024: "obligations for financial guarantees and loan commitments to external creditors EUR 3,370 million ;Undrawn commited credit facilities amount to EUR 10.0 billion. Uncommitted / short-term programme undrawn capacity: European commercial paper: EUR 4.9 billion undrawn (limit EUR 5.0bn; EUR 0.1bn used as of 31-Dec-2024). US commercial paper: US$ 3.0 billion undrawn (limit US$ 3.0bn; not used as of 31-Dec-2024). Debt issuance programme (limit EUR 15.0bn; EUR 6.6bn used as of 31-Dec-2024)",
            2023: "obligations for financial guarantees and loan commitments to external creditors EUR 1,123 million; Uncommitted / short-term programme undrawn capacity: European commercial paper: EUR 4.8 billion undrawn (limit EUR 5.0bn; EUR 0.2bn used as of 31-Dec-2023). US commercial paper: US$ 3.0 billion undrawn (limit US$ 3.0bn; not used as of 31-Dec-2023).Debt issuance programme (limit EUR 15.0bn; EUR 6.1bn used as of 31-Dec-2023)",
            2022: "not provided",
        },
        # Add other KPIs and years as needed
    }

In [ ]:
expected_answers_Walmart={
        "Total Sales Revenue": {
            2025:"USD 680,985 million",
            2024: "USD 648,125 million",
            2023: "USD 611,289 million"
        },
        "Gross Profit": {
            2025:"USD 162,785 million",
            2024: "USD 152,495 million",
            2023: "USD 142,160 million"
        },
        "EBITDA": {
            2025:"USD 36,855 million or USD 42,321 million",
            2024: "USD 38,865 million ",
            2023: "USD 31,373 million"
        },
         "EBIT": {
            2025:"USD 29,348 million",
            2024: "USD 27,012 million",
            2023: "USD 20,428 million"
        },
         "Net Income": {
            2025:"USD 20,157 million or USD 19,436 million",
            2024: "USD 15,511 million or USD 16,270 million",
            2023: "USD 11,292 million or USD 11,680 million"
        },
         "Total Cash": {
            2025:"USD 9.0 billion or USD 9,037 million",
            2024: "USD 9.9 billion or USD 9,867 million",
            2023: "Not provided"
        },
        "Total Equity": {
            2025:"USD 97,421 million or USD 91,013 million",
            2024: "USD 90,349 million or USD 83,861 million",
            2023: "USD 76,693 million"
        },
        "Tangible Net Worth": {
            2025:"USD 64,400 million",
            2024: "USD 58358",
            2023: "Not provied"
        },
        "Capital Expenditures (Capex)": {
            2025:"USD 23,783 million",
            2024: "USD 20,606 million",
            2023: "USD 16,857 million"
        },
        "Free Cash Flow": {
            2025:"USD 12,660 million or USD 12.7 billion",
            2024: "USD 15,120 million or USD 15.1 billion",
            2023: "USD 11,984 million or USD 12.0 billion"
        },
        "Unrestricted Cash": {
            2025:"USD 9.0 billion or USD 9,037 million",
            2024: "USD 9.9 billion or USD 9,867 million",
            2023: "Not provided"
        },
        "Undrawn Loan Facilities": {
            2025:"USD 15,000 million (undrawn)",
            2024: "USD 15,000 million (undrawn)",
            2023: "Not provided"
        },
        # Add other KPIs and years as needed
    }

In [ ]:
# Datasets for quantitative KPI extraction: KPI name, year, client, query, expected_answer
#  RWE
df_quantitative_kpis_RWE = create_quantitative_kpi_dataframe(
    client_company="RWE",
    years=[2024, 2023, 2022],
    expected_answers = expected_answers_RWE

)

# Walmart
df_quantitative_kpis_Walmart = create_quantitative_kpi_dataframe(
    client_company="Walmart",
    years=[2025, 2024, 2023],
    expected_answers=   expected_answers_Walmart
)

# Display RWE DataFrame
#print(df_quantitative_kpis_RWE)

##### ACTION: Set company name and index

In [ ]:
company = "RWE" #### Change to "RWE" or "Walmart"
idx = "0" # index to access

client = "default_client" if company == "RWE" else company 
df_quantitative_kpis = df_quantitative_kpis_Walmart if company == "Walmart" else df_quantitative_kpis_RWE


### Single Agent

In [ ]:
# Run the agent and prepare dataset
dataset_agent_goal = await agent_run_with_history(
    query_list=df_quantitative_kpis["query"].tolist(),
    expected_answers=df_quantitative_kpis["expected_answer"].tolist(),
    expected_kpi_names=df_quantitative_kpis["kpi_name"].tolist(),
    name=f"{company}_single_agent_goal_accuracy_quantitative_{idx}",
    client=client,
    alpha_hybrid=0.6,
)



In [ ]:
# Run the evaluation experiment
result_agent_goal_quantitative = await run_evaluation_agent_goal_accuracy.arun(
    dataset_agent_goal, 
    name=f"{company}_single_agent_goal_accuracy_quantitative_{idx}"
)

In [ ]:
write_csv(sorted(result_agent_goal_quantitative, key=lambda r: (r.get("kpi_name", ""), r.get("year", 0)))
, f"{company}_single_agent_quantitative_speedboat_{idx}.csv")

### Multi Agent

In [ ]:
# Run multi-agent evaluation
dataset_multi_agent = await multi_agent_run_with_history(
    query_list=df_quantitative_kpis["query"].tolist(),
    expected_answers=df_quantitative_kpis["expected_answer"].tolist(),
    expected_kpi_names=df_quantitative_kpis["kpi_name"].tolist(),
    name=f"{company}_multi_agent_goal_accuracy_{idx}",
    client=client,
    alpha_hybrid=0.6,
)


In [ ]:
# Run evaluation
result_multi_agent = await run_evaluation_agent_goal_accuracy.arun(
    dataset_multi_agent, 
    name=f"{company}_multi_agent_goal_accuracy_{idx}"
)

In [ ]:
write_csv(sorted(result_multi_agent, key=lambda r: (r.get("kpi_name", ""), r.get("year", 0)))
, f"{company}_multi_agent_quantitative_speedboat_{idx}.csv")

## EVAL QUALITATIVE (Without Reference)

Evaluate qualitative KPIs using `AgentGoalAccuracyWithoutReference` - no expected answers needed.
The LLM judges whether the agent successfully addressed the user's query based on the conversation.

In [ ]:
# -----------------------------
# Qualitative KPI DataFrame and Evaluation Functions
# -----------------------------

from agent.query_qualtitative_kpi import QualitativeKPIs_speedboat


def create_qualitative_kpi_dataframe(
    client_company: str,
) -> pd.DataFrame:
    """
    Create a DataFrame with qualitative KPIs from the speedboat framework.
    Uses QualitativeKPIs_speedboat class for standardized queries.
    No expected answers needed - will use AgentGoalAccuracyWithoutReference.
    
    Args:
        client_company: The company name (e.g., "RWE", "Walmart")
    
    Returns:
        DataFrame with columns: kpi_name, client, query
    """
    # Build queries using QualitativeKPIs_speedboat
    kpis = QualitativeKPIs_speedboat(client=client_company)
    kpis.build_queries()
    
    # Extract KPI names and queries from the dataclass
    rows = []
    for attr_name in dir(kpis):
        # Skip private attributes and methods
        if attr_name.startswith('_') or attr_name in ('client', 'build_queries', 'all_queries', 'all_queries_list'):
            continue
        
        query = getattr(kpis, attr_name, None)
        if query is not None and isinstance(query, str):
            # Convert attribute name to readable KPI name
            kpi_name = attr_name.replace('_', ' ').title()
            rows.append({
                "kpi_name": kpi_name,
                "client": client_company,
                "query": query,
            })
    
    df = pd.DataFrame(rows)
    print(f"Created Qualitative KPI DataFrame with {len(df)} rows for {client_company}")
    print(f"KPIs: {', '.join(df['kpi_name'].tolist())}")
    return df


@experiment()
async def run_evaluation_agent_goal_accuracy_without_reference(
    row: Dict[str, Any], 
    name: str = "agent_goal_accuracy_no_ref"
) -> Dict[str, Any]:
    """
    Evaluate agent goal accuracy WITHOUT reference for a single row.
    Uses AgentGoalAccuracyWithoutReference - LLM judges if query was addressed.
    """
    print(f"Evaluating: {row.get('kpi_name', '')}")
    
    metric = AgentGoalAccuracyWithoutReference(llm=llm)
    result = await metric.ascore(user_input=row.get("ragas_messages", []))
    
    return {
        "kpi_name": row.get("kpi_name", ""),
        "agent_goal_accuracy": float(result.value),
        "experiment_name": name,
        **row,
    }


async def agent_run_qualitative_with_history(
    query_list: List[str],
    kpi_names: List[str],
    name: str = "qualitative_evaluation",
    client: str = "default_client",
    reranker_entity: str = "RRF",
    reranker_fact: str = "RRF",
    limit_vector_results: int = 15,
    alpha_hybrid: float = 0.6,
    limit_fact: int = 30,
    limit_entity: int = 10,
    k: int = 2,
) -> Dataset:
    """
    Run the SINGLE agent for qualitative KPIs and prepare dataset.
    No expected answers needed.
    
    Returns dataset with: query, ragas_messages, kpi_name, response, value, num_tool_calls
    """
    logger = logging.getLogger(__name__)
    
    try:
        await initialize_graph()
        logger.info("Graph database initialized")
        graph_ok = await test_graph_connection()
        if not graph_ok:
            logger.error("Graph database connection test failed.")
    except Exception as e:
        logger.error(f"Failed to initialize graph database: {e}")
        raise
    
    try:
        initialize_database()
        logger.info("Vector database initialized")
    except Exception as e:
        logger.error(f"Failed to initialize vector database: {e}")
        raise

    agent_runner = single_agent_run_input_list(
        client=client,
        reranker_entity=reranker_entity,
        reranker_fact=reranker_fact,
        limit_vector_results=limit_vector_results,
        alpha_hybrid=alpha_hybrid,
        limit_fact=limit_fact,
        limit_entity=limit_entity,
        k=k,
    )
    
    queries = []
    ragas_messages_list = []
    kpi_names_list = []
    responses = []
    tool_call_counts = []
    
    for query, kpi_name in zip(query_list, kpi_names):
        print(f"Processing: {query[:70]}...")
        result = await agent_runner.run_with_full_history([query], kpi_name=kpi_name)
        
        # Extract input_list for tool call counting
        input_list = result.get("input_list", [])
        num_tool_calls = sum(1 for item in input_list if item.get("type") == "function_call")
        
        # Extract final_output
        final_output = result.get("final_output", {})
        if isinstance(final_output, dict):
            response = (result.get("final_output", {}).get("response", "") or "").replace("\n", " ")
            value = result.get("final_output", {}).get("value", "")
        else:
            response = str(final_output)
            value = ""
        
        if response == "":
            response_agent = value
        else:
            response_agent = response

        ragas_msgs = agent_runner.convert_to_ragas_messages(
            input_list,
            final_output=final_output
        )

        queries.append(query)
        ragas_messages_list.append(ragas_msgs)
        kpi_names_list.append(kpi_name)
        responses.append(response_agent)
        tool_call_counts.append(num_tool_calls)
        
        if ragas_msgs:
            print(f"  -> Final AI message: {str(ragas_msgs[-1])[:200]}...")
            print(f"  -> Tool calls: {num_tool_calls}")
    
    await close_graph()
    close_database()
    logger.info("Databases closed")
    
    # Prepare dataset with additional fields
    dataset = Dataset(
        name=name,
        backend="local/csv",
        root_dir="./data/agent_goal_evaluation"
    )
    
    for q, msgs, kpi_name, resp, num_tools in zip(
        queries, ragas_messages_list, kpi_names_list, responses, tool_call_counts
    ):
        dataset.append({
            "kpi_name": kpi_name,
            "num_tool_calls": num_tools,
            "query": q,
            "response_agent": resp,
            "ragas_messages": msgs,
        })
    
    dataset.save()
    print(f"Qualitative Dataset prepared with {len(queries)} samples")
    return dataset


async def multi_agent_run_qualitative_with_history(
    query_list: List[str],
    kpi_names: List[str],
    name: str = "multi_qualitative_evaluation",
    client: str = "default_client",
    reranker_entity: str = "RRF",
    reranker_fact: str = "RRF",
    limit_vector_results: int = 15,
    alpha_hybrid: float = 0.6,
    limit_fact: int = 30,
    limit_entity: int = 10,
    k: int = 2,
) -> Dataset:
    """
    Run the MULTI-AGENT for qualitative KPIs and prepare dataset.
    No expected answers needed.
    
    Returns dataset with: query, ragas_messages, kpi_name, response, value, num_tool_calls
    """
    logger = logging.getLogger(__name__)
    
    try:
        await initialize_graph()
        logger.info("Graph database initialized")
        graph_ok = await test_graph_connection()
        if not graph_ok:
            logger.error("Graph database connection test failed.")
    except Exception as e:
        logger.error(f"Failed to initialize graph database: {e}")
        raise
    
    try:
        initialize_database()
        logger.info("Vector database initialized")
    except Exception as e:
        logger.error(f"Failed to initialize vector database: {e}")
        raise

    agent_runner = multi_agent_router_orchestrator_run_input_list(
        client=client,
        reranker_entity=reranker_entity,
        reranker_fact=reranker_fact,
        limit_vector_results=limit_vector_results,
        alpha_hybrid=alpha_hybrid,
        limit_fact=limit_fact,
        limit_entity=limit_entity,
        k=k,
    )
    
    queries = []
    ragas_messages_list = []
    kpi_names_list = []
    responses = []
    tool_call_counts = []
    i = 1
    
    for query, kpi_name in zip(query_list, kpi_names):
        print(f"{i} Processing: {query[:70]}...")
        result = await agent_runner.run_with_full_history([query], kpi_name)
        
        # Extract input_list for tool call counting
        input_list = result.get("input_list", [])
        num_tool_calls = sum(1 for item in input_list if item.get("type") == "function_call")
        
        # Extract final_output
        final_output = result.get("final_output", {})
        if isinstance(final_output, dict):
            response_agent = (result.get("final_output", {}).get("response", "") or "").replace("\n", " ")

        else:
            response_agent = str(final_output)

        
        ragas_msgs = agent_runner.convert_to_ragas_messages(
            input_list,
            final_output=final_output
        )
        
        

        queries.append(query)
        ragas_messages_list.append(ragas_msgs)
        kpi_names_list.append(kpi_name)
        responses.append(response_agent)
        tool_call_counts.append(num_tool_calls)
        
        if ragas_msgs:
            print(f"  -> Final AI message: {str(ragas_msgs[-1])[:200]}...")
            print(f"  -> Tool calls: {num_tool_calls}, Response: {response_agent[:100]}...")
            print("-----------------------\n")
        i += 1
        
    await close_graph()
    close_database()
    logger.info("Databases closed")
    
    # Prepare dataset with additional fields
    dataset = Dataset(
        name=name,
        backend="local/csv",
        root_dir="./data/agent_goal_evaluation"
    )
    
    for q, msgs, kpi_name, resp, num_tools in zip(
        queries, ragas_messages_list, kpi_names_list, responses, tool_call_counts
    ):
        dataset.append({
            "kpi_name": kpi_name,
            "num_tool_calls": num_tools,
            "query": q,
            "response_agent": resp,
            "ragas_messages": msgs,
            
            
        })
    
    dataset.save()
    print(f"Multi-Agent Qualitative Dataset prepared with {len(queries)} samples")
    return dataset



In [ ]:
# Create qualitative KPI dataframe for RWE using QualitativeKPIs_speedboat
df_qualitative_kpis_RWE = create_qualitative_kpi_dataframe(
    client_company="RWE",
)

df_qualitative_kpis_Walmart = create_qualitative_kpi_dataframe(
    client_company="Walmart",
)


In [ ]:
client_name="Walmart"

idx = 0

client = client_name

if client_name == "Walmart":
    df = df_qualitative_kpis_Walmart
elif client_name == "RWE":
    df = df_qualitative_kpis_RWE

### Single Agent - Qualitative

In [ ]:
# Run single agent for qualitative KPIs
dataset_qualitative_single = await agent_run_qualitative_with_history(
    query_list=df["query"].tolist(),
    kpi_names=df["kpi_name"].tolist(),
    name=f"{client_name}_qualitative_single_agent_{idx}",
    client=client,
    alpha_hybrid=0.6,
)

In [ ]:
# Run evaluation (without reference)
result_qualitative_single = await run_evaluation_agent_goal_accuracy_without_reference.arun(
    dataset_qualitative_single, 
    name=f"{client_name}_qualitative_single_agent_goal_accuracy_{idx}"
)
write_csv(sorted(result_qualitative_single, key=lambda r: r.get("kpi_name", "")), f"{client_name}_qualitative_single_agent_goal_accuracy_{idx}.csv")

### Multi Agent - Qualitative

In [ ]:
# Run multi-agent for qualitative KPIs
dataset_qualitative_multi = await multi_agent_run_qualitative_with_history(
    query_list=df["query"].tolist(),
    kpi_names=df["kpi_name"].tolist(),
    name=f"{client_name}_qualitative_multi_agent_{idx}",
    client=client,
    alpha_hybrid=0.6,
)

In [ ]:
# Run evaluation (without reference)
result_qualitative_multi = await run_evaluation_agent_goal_accuracy_without_reference.arun(
    dataset_qualitative_multi, 
    name=f"{client_name}_qualitative_multi_agent_goal_accuracy"
)
write_csv(sorted(result_qualitative_multi, key=lambda r: r.get("kpi_name", "")), f"{client_name}_qualitative_multi_agent_goal_accuracy_{idx}.csv")

###  Single vs Multi Agent 

Load experiment results from CSV files and compare agent performance.

In [ ]:
# -----------------------------
# Load Qualitative Experiment Data from CSV Files
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Define experiment file paths
EXPERIMENTS_DIR = "./data/agent_goal_accuracy/experiments"

# RWE files
rwe_single_file = f"{EXPERIMENTS_DIR}/RWE_qualitative_single_agent_goal_accuracy.csv"
rwe_multi_file = f"{EXPERIMENTS_DIR}/RWE_qualitative_multi_agent_goal_accuracy.csv"

# Walmart files
walmart_single_file = f"{EXPERIMENTS_DIR}/Walmart_qualitative_single_agent_goal_accuracy.csv"
walmart_multi_file = f"{EXPERIMENTS_DIR}/Walmart_qualitative_multi_agent_goal_accuracy.csv"

# Load RWE data
df_rwe_single = pd.read_csv(rwe_single_file)
df_rwe_single["agent_type"] = "Single"
df_rwe_single["company"] = "RWE"

df_rwe_multi = pd.read_csv(rwe_multi_file)
df_rwe_multi["agent_type"] = "Multi"
df_rwe_multi["company"] = "RWE"

# Load Walmart data
df_walmart_single = pd.read_csv(walmart_single_file)
df_walmart_single["agent_type"] = "Single"
df_walmart_single["company"] = "Walmart"

df_walmart_multi = pd.read_csv(walmart_multi_file)
df_walmart_multi["agent_type"] = "Multi"
df_walmart_multi["company"] = "Walmart"

# Combine all data
df_rwe = pd.concat([df_rwe_single, df_rwe_multi], ignore_index=True)
df_walmart = pd.concat([df_walmart_single, df_walmart_multi], ignore_index=True)
df_all = pd.concat([df_rwe, df_walmart], ignore_index=True)

print(f". Loaded experiment data:")
print(f"   RWE Single:     {len(df_rwe_single)} samples")
print(f"   RWE Multi:      {len(df_rwe_multi)} samples")
print(f"   Walmart Single: {len(df_walmart_single)} samples")
print(f"   Walmart Multi:  {len(df_walmart_multi)} samples")
print(f"   Total:          {len(df_all)} samples")

# Tool Call F1 + Agent Goal Accuracy for Derived KPIs

## Overview
This evaluation framework assesses agentic RAG systems on their ability to compute **derived KPIs** - financial metrics that require combining multiple atomic values from annual reports. Unlike atomic KPIs (e.g., "Revenue for 2023") which can be directly retrieved, derived KPIs demand:
1. **Tool orchestration**: Identifying and executing the correct calculation tool (`kpi_calculator`)
2. **Information retrieval**: Searching for and extracting required input values (e.g., revenue, COGS)
3. **Computational accuracy**: Performing the correct calculation
4. **Answer synthesis**: Formulating a natural language response with context

## Dual Evaluation Metrics

### Tool Call F1 Score
$$F_{1,\text{tool}} = 0.7 \times S_{\text{required}} + 0.3 \times S_{\text{search}}$$

**Purpose**: Measures the agent's **procedural proficiency** in tool usage and information gathering.

**Components**:
- $S_{\text{required}}$ (70% weight): Binary score for using mandatory tools
  - For derived KPIs: `kpi_calculator` is required
  - Ensures agent recognizes when computation is needed vs. direct retrieval
- $S_{\text{search}}$ (30% weight): Search coverage score
  - Measures how well the agent retrieves necessary input values
  - Based on semantic search tool calls to find atomic KPIs

**Rationale for weights**: Required tool usage is weighted higher because it represents the fundamental capability distinction - an agent that cannot identify when to compute vs. retrieve has failed at task classification.

### Agent Goal Accuracy
$$A_{\text{goal}} = \text{AgentGoalAccuracyWithReference}(\text{response}, \text{expected\_answer})$$

**Purpose**: Measures the **semantic correctness** of the final answer using LLM-based evaluation.

**Mechanism**: 
- Uses RAGAS `AgentGoalAccuracyWithReference` metric
- GPT-4 evaluates whether the agent's natural language response correctly answers the query
- Considers both numerical accuracy and contextual explanation quality
- Reference-based: Compares against ground truth expected answers

**Example evaluation**:
- Query: "What was Walmart's Gross Profit in 2023?"
- Agent response: "Gross Profit for 2023 was $147.6B (Revenue: $611.3B - COGS: $463.7B)"
- Expected: "$147.6 billion"
- Evaluation: Checks numerical match + explanation coherence

### Combined Score
$$S_{\text{combined}} = 0.4 \times F_{1,\text{tool}} + 0.6 \times A_{\text{goal}}$$

**Purpose**: Holistic agent performance metric balancing process and outcome.

**Rationale for weights**:
- **60% Goal Accuracy**: Prioritizes end-user value - correctness of the final answer
- **40% Tool F1**: Ensures process quality - proper methodology even if answer is correct by chance
- Philosophy: "Right answer through right process" is valued over lucky guesses


## Evaluation Workflow
1. **Agent execution**: Process query through single/multi-agent architecture
2. **Tool tracking**: Record all tool calls (search, calculator) during execution
3. **Tool F1 computation**: Analyze tool usage against requirements
4. **Response extraction**: Extract natural language answer from `final_output["response"]`
5. **RAGAS evaluation**: Compare response against expected answer using LLM judge
6. **Combined scoring**: Weight and combine both metrics

In [ ]:
from ragas import Dataset, experiment
from ragas.messages import ToolCall
from ragas.metrics.collections import AgentGoalAccuracyWithReference
import re

# -----------------------------
# Tool Call F1 + Agent Goal Accuracy Evaluation for Derived KPIs
# -----------------------------

# Define EXPECTED TOOL PATTERNS for each derived KPI
DERIVED_KPI_TOOL_REQUIREMENTS_RWE = {
    "Gross Profit": {
        "required_tools": ["kpi_calculator"],
        "search_min": 1,
        "description": "Search for revenue/cogs components, then calculate"
    },
    "Tangible Net Worth": {
        "required_tools": ["kpi_calculator"],
        "search_min": 3,
        "description": "Search for assets/liabilities/intangibles, then calculate"
    },
    "EBITDA": {
        "required_tools": [],
        "search_min": 1,
        "description": "Search for EBITDA or calculate from EBIT + D&A"
    },
}


DERIVED_KPI_TOOL_REQUIREMENTS_Walmart = {
    "Gross Profit": {
        "required_tools": [],
        "search_min": 1,
        "description": "Search for revenue/cogs components, then calculate"
    },
    "Tangible Net Worth": {
        "required_tools": ["kpi_calculator"],
        "search_min": 3,
        "description": "Search for assets/liabilities/intangibles, then calculate"
    },
    "EBITDA": {
        "required_tools": ["kpi_calculator"],
        "search_min": 2,
        "description": "Search for EBITDA or calculate from EBIT + D&A"
    },
}

def extract_tool_call_names(input_list: List[Dict[str, Any]]) -> List[str]:
    """Extract just the tool call names from input_list."""
    names = []
    for item in input_list:
        if item.get("type") == "function_call":
            names.append(item.get("name", ""))
    return names


def compute_tool_call_f1_flexible(
    actual_names: List[str], 
    kpi_name: str,
    derived_kpi_tool_requirements: Dict[str, Any],
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Compute a flexible Tool Call F1 based on key tool requirements.
    
    F1 = 0.7 × required_tool_score + 0.3 × search_score
    """
    requirements = derived_kpi_tool_requirements.get(kpi_name, {
        "required_tools": [],
        "search_min": 1,
    })
    
    required_tools = requirements["required_tools"]
    search_min = requirements["search_min"]
    
    # Count tool types
    search_count = sum(1 for name in actual_names if "search" in name.lower())
    calculator_called = "kpi_calculator" in actual_names
    
    # Score required tools
    if required_tools:
        required_called = sum(1 for tool in required_tools if tool in actual_names)
        required_tool_score = required_called / len(required_tools)
    else:
        required_tool_score = 1.0
    
    # Score search coverage
    if search_min > 0:
        search_score = min(1.0, search_count / search_min)
    else:
        search_score = 1.0 if search_count > 0 else 0.5
    
    # Combined F1 = 0.7 × required_tool_score + 0.3 × search_score
    f1 = 0.7 * required_tool_score + 0.3 * search_score
    
    return {
        "f1": f1,
        "required_tool_score": required_tool_score,
        "search_score": search_score,
        "calculator_called": calculator_called,
        "search_count": search_count,
    }


async def agent_run_with_tool_call_tracking(
    query_list: List[str],
    kpi_names: List[str],
    years: List[Any],
    expected_answers: Optional[List[str]] = None,
    client: str = "default_client",
    name: str = "tool_call_evaluation",
    alpha_hybrid: float = 0.8,
    agent_type: str = "single",  # NEW: "single" or "multi"
    **kwargs
) -> List[Dict[str, Any]]:
    """
    Run the agent for derived KPIs and return results list.
    Supports both single agent and multi-agent.
    
    Args:
        agent_type: "single" for single agent, "multi" for multi-agent router
    """
    logger = logging.getLogger(__name__)
    
    # Default to empty strings if no expected answers provided
    if expected_answers is None:
        expected_answers = [""] * len(query_list)
    
    try:
        await initialize_graph()
        initialize_database()
    except Exception as e:
        logger.error(f"Failed to initialize databases: {e}")
        raise

    # Create agent runner based on type
    if agent_type == "multi":
        agent_runner = multi_agent_router_orchestrator_run_input_list(
            client=client,
            alpha_hybrid=alpha_hybrid,
            **kwargs
        )
        print(f" Using MULTI-AGENT Router")
    else:
        agent_runner = single_agent_run_input_list(
            client=client,
            alpha_hybrid=alpha_hybrid,
            **kwargs
        )
        print(f" Using SINGLE Agent")
    
    results_list = []
    
    for query, kpi_name, year, expected in zip(query_list, kpi_names, years, expected_answers):
        print(f"Processing: {query[:60]}...")
        result = await agent_runner.run_with_full_history([query], f"{kpi_name}_{year}")
        
        input_list = result.get("input_list", [])
        actual_tool_names = extract_tool_call_names(input_list)
        
        # Extract agent's final answer
        final_output = result.get("final_output", {})
        agent_answer = final_output.get("value", "") if isinstance(final_output, dict) else str(final_output)
        
        # Convert to RAGAS messages format for AgentGoalAccuracy
        ragas_msgs = agent_runner.convert_to_ragas_messages(
            input_list,
            final_output=final_output
        )
        
        results_list.append({
            "query": query,
            "kpi_name": kpi_name,
            "year": year,
            "client": client,
            "agent_type": agent_type,
            "input_list": input_list,
            "actual_tool_names": actual_tool_names,
            "agent_answer": agent_answer,
            "expected_answer": expected,
            "ragas_messages": ragas_msgs,  # For AgentGoalAccuracy
            "final_output": final_output,
        })
    
    await close_graph()
    close_database()
    
    print(f"Collected {len(results_list)} samples for evaluation")
    return results_list


async def run_tool_call_f1_and_goal_accuracy_evaluation(
    results_list: List[Dict[str, Any]],
    derived_kpi_tool_requirements: Dict[str, Any],
    llm,  # RAGAS LLM instance
    name: str = "tool_call_goal_accuracy",

) -> List[Dict[str, Any]]:
    """
    Evaluate both Tool Call F1 AND Agent Goal Accuracy (using RAGAS).
    
    Returns combined score:
    - tool_call_f1: Did the agent use the right workflow? (F1 = 0.7 × required_tool_score + 0.3 × search_score)
    - agent_goal_accuracy: Did it achieve the goal? (RAGAS AgentGoalAccuracyWithReference)
    - combined_score: Weighted combination
    """
    # Initialize RAGAS metric for goal accuracy
    goal_accuracy_metric = AgentGoalAccuracyWithReference(llm=llm)
    
    evaluation_results = []
    
    for i, row in enumerate(results_list):
        kpi_name = row.get("kpi_name", "")
        year = row.get("year", "")
        actual_names = row.get("actual_tool_names", [])
        ragas_messages = row.get("ragas_messages", [])
        expected_answer = row.get("expected_answer", "")
        agent_answer = row.get("agent_answer", "")
        
        print(f"Evaluating {i+1}/{len(results_list)}: {kpi_name} ({year})...")
        
        # 1. Compute Tool Call F1
        tool_metrics = compute_tool_call_f1_flexible(actual_names, kpi_name, derived_kpi_tool_requirements)
        
        # 2. Compute Agent Goal Accuracy using RAGAS
        try:
            goal_result = await goal_accuracy_metric.ascore(
                user_input=ragas_messages,
                reference=expected_answer
            )
            goal_accuracy = float(goal_result.value) if hasattr(goal_result, 'value') else float(goal_result)
        except Exception as e:
            print(f"   Error computing AgentGoalAccuracy: {e}")
            goal_accuracy = 0.0
        
        # 3. Combined score: 40% tool usage + 60% goal accuracy
        combined_score = 0.4 * tool_metrics["f1"] + 0.6 * goal_accuracy
        
        # Status emoji
        status = "" if combined_score >= 0.8 else "" if combined_score >= 0.5 else ""
        print(f"  {status} F1={tool_metrics['f1']:.2f}, GoalAcc={goal_accuracy:.2f}, Combined={combined_score:.2f}")
        
        evaluation_results.append({
            "kpi_name": kpi_name,
            "year": year,
            "query": row.get("query", ""),
            "agent_type": row.get("agent_type", "single"),
            # Tool Call Metrics
            "tool_call_f1": tool_metrics["f1"],
            "required_tool_score": tool_metrics["required_tool_score"],
            "search_score": tool_metrics["search_score"],
            "calculator_called": tool_metrics["calculator_called"],
            "search_count": tool_metrics["search_count"],
            # Goal Accuracy (RAGAS)
            "agent_goal_accuracy": goal_accuracy,
            # Combined
            "combined_score": combined_score,
            # Raw data
            "agent_answer": agent_answer[:300] if agent_answer else "",
            "expected_answer": expected_answer[:300] if expected_answer else "",
            "actual_tool_names": actual_names,
            "num_actual_calls": len(actual_names),
            "experiment_name": name,
        })
    
    return evaluation_results



# for kpi, reqs in DERIVED_KPI_TOOL_REQUIREMENTS.items():
#     print(f"  • {kpi}: required={reqs['required_tools']}, min_searches={reqs['search_min']}")

#### SETTINGS

In [ ]:
company = "Walmart"  #  "RWE" or "Walmart"

if company == "RWE": client = "default_client"
else: client = company

#df_derived_kpis = df_derived_kpis_RWE if company == "RWE" else df_derived_kpis_Walmart
tool_requirements =  DERIVED_KPI_TOOL_REQUIREMENTS_RWE if company == "RWE" else DERIVED_KPI_TOOL_REQUIREMENTS_Walmart

In [ ]:
import pandas as pd

# Expected answers for derived KPIs
# Use "or" to specify multiple acceptable values (same format as expected_answers_RWE)
# The AgentGoalAccuracyWithReference metric will accept any of the listed values
EXPECTED_ANSWERS_DERIVED_KPIS_RWE = {
    "Gross Profit": {
        # Gross Profit = Revenue - COGS (Cost of Materials)
        # Multiple values due to different revenue/COGS definitions in reports
        2024: "EUR 9,031 million or EUR 8,816 million or EUR 15,408 million",
        2023: "EUR 11,362 million or EUR 11,530 million or EUR 17,327 million",
        2022: "Not provided"
    },
    "Tangible Net Worth": {
        # Tangible Net Worth = Total Equity - Intangible Assets
        # Multiple values depending on equity figure used
        2024: "EUR 23,373 million or EUR 23,363 million",
        2023: "EUR 23,817 million or EUR 23,854 million",
        2022: "Not provided"
    },
    "Unrestricted Cash": {
        # Unrestricted Cash = Total Cash - Restricted Cash
        2024: "EUR 5,090 million or not provided",
        2023: "EUR 6,917 million or not provided",
        2022: "Not provided"
    },
    "EBITDA": {
        # EBITDA directly from reports or calculated
        2024: "EUR 5,680 million",
        2023: "EUR 7,749 million",
        2022: "EUR 6,310 million",
    },
}


EXPECTED_ANSWERS_DERIVED_KPIS_Walmart = {
 "EBITDA": {
            2025:"USD 36,855 million or USD 42,321 million",
            2024: "USD 38,865 million ",
            2023: "USD 31,373 million"
        },

        "Tangible Net Worth": {
            2025:"USD 64,400 million or 93,192 million" ,
            2024: "USD 58,358 million or USD 86,471 million",
            2023: "Not provied"
        },
        "Unrestricted Cash": {
            2025:"USD 9.0 billion or USD 9,037 million",
            2024: "USD 9.9 billion or USD 9,867 million",
            2023: "Not provided"
        }
}

if company == "RWE": expected_answers = EXPECTED_ANSWERS_DERIVED_KPIS_RWE
elif company == "Walmart": expected_answers = EXPECTED_ANSWERS_DERIVED_KPIS_Walmart

years = [2024, 2023, 2022] if company == "RWE" else [2025, 2024, 2023]

def create_derived_kpi_dataframe(
    client_company: str,
    years: List[int],
    expected_answers: Optional[Dict[str, Dict[int, str]]] = None,
) -> pd.DataFrame:
    """
    Create a DataFrame for derived KPIs that require kpi_calculator.
    
    Expected answers can contain multiple acceptable values separated by " or ".
    Example: "EUR 9,031 million or EUR 8,816 million"
    """

    if client_company == "RWE":
        derived_kpi_names = [
            "Gross Profit",
            "Tangible Net Worth",
            # "Unrestricted Cash",
            # "EBITDA",
        ]
    elif client_company == "Walmart":
        derived_kpi_names = [
            #"Gross Profit",
            "Tangible Net Worth",
            # "Unrestricted Cash",
             "EBITDA",
        ]
    
    # Use provided expected answers or defaults
    if expected_answers is None:
        expected_answers = EXPECTED_ANSWERS_DERIVED_KPIS_RWE
    
    rows = []
    for kpi in derived_kpi_names:
        for year in years:
            expected = ""
            if kpi in expected_answers and year in expected_answers[kpi]:
                expected = expected_answers[kpi][year]
            
            rows.append({
                "kpi_name": kpi,
                "year": year,
                "client": client_company,
                "query": f"Provide the {kpi} for the fiscal year {year} for {client_company}.",
                "expected_answer": expected,
            })
    
    df = pd.DataFrame(rows)
    print(f"Created Derived KPI DataFrame with {len(df)} rows for {client_company}")
    print(f"  → Expected answers provided for {sum(1 for r in rows if r['expected_answer'])}/{len(rows)} queries")
    return df


# Create derived KPI dataframe for RWE with expected answers
df_derived_kpis = create_derived_kpi_dataframe(
    client_company=company,
    years=years,
    expected_answers=expected_answers
)


#### Single-Agent Evaluation

In [ ]:
# -----------------------------
# SINGLE AGENT: Tool Call + Goal Accuracy Evaluation
# -----------------------------

# Run single agent for derived KPIs (with expected answers and RAGAS messages)
dataset_single_agent = await agent_run_with_tool_call_tracking(
    query_list=df_derived_kpis["query"].tolist(),
    kpi_names=df_derived_kpis["kpi_name"].tolist(),
    years=df_derived_kpis["year"].tolist(),
    expected_answers=df_derived_kpis["expected_answer"].tolist(),
    client=client,
    name=f"{company}_single_agent_derived_kpis",
    alpha_hybrid=0.7,
    agent_type="single",  # Single agent
)

In [ ]:
# Run the combined evaluation (Tool Call F1 + Agent Goal Accuracy)
result_single_agent = await run_tool_call_f1_and_goal_accuracy_evaluation(
    dataset_single_agent,
    tool_requirements,
    llm=llm,  # Pass the RAGAS LLM instance
    name=f"{company}_single_agent_tool_f1_goal_accuracy_2"
)

### Multi-Agent Evaluation

In [ ]:
# -----------------------------
# MULTI-AGENT: Tool Call + Goal Accuracy Evaluation
# -----------------------------

# Run multi-agent for derived KPIs (with expected answers and RAGAS messages)
dataset_multi_agent = await agent_run_with_tool_call_tracking(
    query_list=df_derived_kpis["query"].tolist(),
    kpi_names=df_derived_kpis["kpi_name"].tolist(),
    years=df_derived_kpis["year"].tolist(),
    expected_answers=df_derived_kpis["expected_answer"].tolist(),
    client=client,
    name=f"{company}_multi_agent_derived_kpis",
    alpha_hybrid=0.6,
    agent_type="multi",  # Multi-agent router
)

In [ ]:
# Run the combined evaluation for Multi-Agent
result_multi_agent = await run_tool_call_f1_and_goal_accuracy_evaluation(
    dataset_multi_agent,
    llm=llm,
    name=f"{company}_multi_agent_tool_f1_goal_accuracy",
    derived_kpi_tool_requirements=tool_requirements
)

In [ ]:
# Print Multi-Agent Results
#df_multi = print_evaluation_results(result_multi_agent, "MULTI Agent: Tool Call F1 + Goal Accuracy")
write_csv(result_multi_agent, f"{company}_multi_agent_derived_f1_goal")

In [ ]:
# -----------------------------
# COMPARISON: Single Agent vs Multi-Agent
# -----------------------------

print("="*80)
print(" COMPARISON: Single Agent vs Multi-Agent")
print("="*80)

# Combine results for comparison
df_single["agent_type"] = "single"
df_multi["agent_type"] = "multi"
df_combined = pd.concat([df_single, df_multi], ignore_index=True)

# Overall comparison
print("\n Overall Metrics Comparison:")
print("-"*60)
print(f"{'Metric':<25} {'Single Agent':>15} {'Multi-Agent':>15}")
print("-"*60)

for metric in ["combined_score", "tool_call_f1", "agent_goal_accuracy"]:
    single_val = df_single[metric].mean()
    multi_val = df_multi[metric].mean()
    diff = multi_val - single_val
    winner = "" if diff > 0 else ("  " if diff == 0 else "")
    print(f"{metric:<25} {single_val:>15.3f} {multi_val:>14.3f} {winner}")

print("-"*60)

# Calculator usage comparison
single_calc = df_single["calculator_called"].mean() * 100
multi_calc = df_multi["calculator_called"].mean() * 100
print(f"{'Calculator Usage (%)':<25} {single_calc:>14.0f}% {multi_calc:>13.0f}%")

# Per-KPI comparison
print(f"\n{'='*80}")
print(" Per-KPI Comparison (Combined Score):")
print("-"*60)
print(f"{'KPI':<25} {'Single':>12} {'Multi':>12} {'Diff':>12}")
print("-"*60)

for kpi in df_combined["kpi_name"].unique():
    single_kpi = df_single[df_single["kpi_name"] == kpi]["combined_score"].mean()
    multi_kpi = df_multi[df_multi["kpi_name"] == kpi]["combined_score"].mean()
    diff = multi_kpi - single_kpi
    winner = "" if diff > 0.05 else ("  " if diff >= -0.05 else "")
    print(f"{kpi:<25} {single_kpi:>12.3f} {multi_kpi:>12.3f} {diff:>+11.3f} {winner}")

print("-"*60)